In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'streamer_drift_diffusion').is_dir():
    ROOT = ROOT / 'pyaxistreamer-master'
if not (ROOT / 'streamer_drift_diffusion').is_dir():
    raise RuntimeError('streamer_python またはその親フォルダを作業ディレクトリにしてください')
sys.path.insert(0, str(ROOT))
from streamer_drift_diffusion import SimulationConfig, run_simulation, load_voltage_waveform, applied_voltage
import fipy
print('Python:', sys.executable)
print('FiPy:', fipy.__version__)

Python: c:\Users\daiko\Documents\Python_venvSystem\3.14system\PyAxiStreamer\env_pyaxistreamer\Scripts\python.exe
FiPy: 4.0.3


In [2]:
# 電場・速度はスタガード配置。CSVのEx/Ey/absEはセル中心 [Td]。
# 計算条件は、このSimulationConfigにまとめて指定する。
config = SimulationConfig(
    nr=400, nz=300, delta=5e-6, electrode_gap=1e-3,
    electron_diffusion=0.0,  # 検証用の一定係数 [m^2/s]。0は拡散なし。
    voltage_mode='impulse', voltage=4000.0,
    impulse_peak_voltage=6000.0, impulse_time_scale=0.25,
    adaptive_timestep=True, t_end=5.001e-9,
    output_interval=1e-10, dt_max=1e-12,
    max_steps=20_000,
)
OUTPUT_DIR = ROOT / ('outputdata_adaptive_staggered' if config.adaptive_timestep else 'outputdata_staggered')
print(config)

SimulationConfig(nr=400, nz=300, delta=5e-06, temperature=300.0, avogadro=6.022141e+23, gas_constant=0.08205736608096, elementary_charge=1.602176462e-19, vacuum_permittivity=8.85e-12, oxygen_fraction=0.2, nitrogen_fraction=0.8, cathode_radius=0.0003, voltage=4000.0, voltage_mode='impulse', impulse_peak_voltage=6000.0, impulse_time_scale=0.25, voltage_waveform_file='C:\\Users\\daiko\\Documents\\Python_venvSystem\\3.14system\\PyAxiStreamer\\pyaxistreamer-master\\streamer_drift_diffusion\\data\\impulse_waveform_V50_tr5_tw6.txt', dt=1e-12, nmax=1e+19, z0=0.0009, sr0=0.0001, sz0=0.0001, background_ionization=10000000000000.0, poisson_tolerance=1e-10, adaptive_timestep=True, t_end=5.001e-09, dt_max=1e-12, dt_min=1e-18, dt_safety_factor=0.5, dt_cfl_number=0.8, dt_chemistry_fraction=0.1, dt_growth_factor=1.2, max_step_retries=12, max_steps=20000, output_interval=1e-10, electrode_gap=0.001, electron_diffusion=0.0)


In [3]:
state = run_simulation(config, output_dir=OUTPUT_DIR)
print('CSV:', OUTPUT_DIR)
print('maximum electron density [m^-3]:', state.ne.max())

step = 104, Time = 0.100000 ns, dt = 5.000000e-13 s
step = 208, Time = 0.200000 ns, dt = 5.000000e-13 s
step = 312, Time = 0.300000 ns, dt = 5.000000e-13 s
step = 416, Time = 0.400000 ns, dt = 5.000000e-13 s
step = 520, Time = 0.500000 ns, dt = 5.000000e-13 s
step = 624, Time = 0.600000 ns, dt = 5.000000e-13 s
step = 728, Time = 0.700000 ns, dt = 5.000000e-13 s
step = 832, Time = 0.800000 ns, dt = 5.000000e-13 s
step = 936, Time = 0.900000 ns, dt = 5.000000e-13 s
step = 1040, Time = 1.000000 ns, dt = 5.000000e-13 s
step = 1144, Time = 1.100000 ns, dt = 9.367935e-13 s
step = 1256, Time = 1.200000 ns, dt = 4.337295e-13 s
step = 1368, Time = 1.300000 ns, dt = 5.450264e-13 s
step = 1480, Time = 1.400000 ns, dt = 5.365387e-13 s
step = 1592, Time = 1.500000 ns, dt = 5.654330e-13 s
step = 1706, Time = 1.600000 ns, dt = 2.842325e-13 s
step = 1841, Time = 1.700000 ns, dt = 5.450394e-13 s
step = 2065, Time = 1.800000 ns, dt = 2.989928e-13 s
step = 2651, Time = 1.900000 ns, dt = 3.464427e-14 s


KeyboardInterrupt: 

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
fields = [(state.ne, 'Electron density [m$^{-3}$]'),
          (state.phi, 'Potential [V]'), (state.abs_e, '|E|/N [Td]')]
for ax, (values, title) in zip(axes, fields):
    image = ax.pcolormesh(state.rh * 1e3, state.zh * 1e3, values.T, shading='nearest', cmap='jet')
    ax.set(xlabel='r [mm]', ylabel='z [mm]', title=title)
    fig.colorbar(image, ax=ax)
plt.show()
# 必要な場合: fig.savefig(ROOT / 'final_state.png', dpi=160)

電位・電場は最終stepの密度更新前、電子密度は更新後の値です。Julia版の既存描画Notebookも、入力先をこの `OUTPUT_DIR` に変更して使用できます。